# t-Tests

## Overview

Compares means for approximately normal data.

| Test | scipy |
|---|---|
| One-sample | `ttest_1samp` |
| Two-sample Welch | `ttest_ind(equal_var=False)` |
| Paired | `ttest_rel` |

Always use Welch for two-sample comparisons. Always report Cohen's d and 95% CI alongside p.

---

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
n_ctrl, n_rest = 65, 68
ctrl = rng.normal(18.0, 5.0, n_ctrl)
rest = rng.normal(20.5, 5.3, n_rest)
before = rng.normal(17.0, 4.5, 50)
after  = before + rng.normal(2.8, 2.0, 50)
print(f"Control:  mean={ctrl.mean():.2f}, SD={ctrl.std():.2f}")
print(f"Restored: mean={rest.mean():.2f}, SD={rest.std():.2f}")

---
## One-Sample t-Test

In [ ]:
t, p = stats.ttest_1samp(ctrl, popmean=17)
ci = stats.t.interval(0.95, df=n_ctrl-1, loc=ctrl.mean(), scale=stats.sem(ctrl))
print(f"One-sample vs mu=17: t={t:.3f}, p={p:.4f}")
print(f"  95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]")

---
## Two-Sample Welch t-Test

In [ ]:
t, p = stats.ttest_ind(rest, ctrl, equal_var=False)
d = (rest.mean()-ctrl.mean()) / np.sqrt((ctrl.std()**2+rest.std()**2)/2)
diff = rest.mean()-ctrl.mean()
se = np.sqrt(ctrl.var(ddof=1)/n_ctrl + rest.var(ddof=1)/n_rest)
df_w = se**4/((ctrl.var(ddof=1)/n_ctrl)**2/(n_ctrl-1)+(rest.var(ddof=1)/n_rest)**2/(n_rest-1))
ci = stats.t.interval(0.95, df=df_w, loc=diff, scale=se)
print(f"Welch: t={t:.3f}, p={p:.4f}")
print(f"  diff={diff:.2f}, 95% CI: [{ci[0]:.2f},{ci[1]:.2f}], d={d:.3f}")
fig, ax = plt.subplots(figsize=(7,4))
ax.violinplot([ctrl, rest], positions=[1,2], showmedians=True)
ax.set_xticks([1,2]); ax.set_xticklabels(["Control","Restored"])
ax.set_ylabel("Species richness")
ax.set_title(f"Richness by Treatment  (d={d:.2f}, p={p:.3f})")
plt.tight_layout(); plt.show()

---
## Paired t-Test

In [ ]:
t, p = stats.ttest_rel(after, before)
dp = after - before
ci = stats.t.interval(0.95, df=len(dp)-1, loc=dp.mean(), scale=stats.sem(dp))
print(f"Paired: t={t:.3f}, p={p:.4f}, dz={dp.mean()/dp.std():.3f}")
print(f"  95% CI on difference: [{ci[0]:.2f}, {ci[1]:.2f}]")
t_w, p_w = stats.ttest_ind(after, before, equal_var=False)
print(f"Wrong (independent): p={p_w:.4f}  <- ignores pairing, less powerful")

---
## Q-Q Assumptions Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, data, lbl in zip(axes, [ctrl, rest], ["Control","Restored"]):
    (osm,osr),(sl,ic,r) = stats.probplot(data, dist="norm")
    ax.scatter(osm, osr, s=20, alpha=0.6, color="steelblue")
    ax.plot(osm, sl*np.array(osm)+ic, color="#e74c3c", lw=1.5)
    ax.set_title(f"Q-Q {lbl}  (r={r:.3f})")
plt.tight_layout(); plt.show()
lev, p_lev = stats.levene(ctrl, rest)
print(f"Levene: F={lev:.3f}, p={p_lev:.4f}  -- Welch valid regardless")

---

## Common Pitfalls

**1. Using Student's t-test (equal variances) by default**  
Welch's t-test is valid whether or not variances are equal and has virtually no power cost when they are. Always use `equal_var=False`.

**2. Reporting only the p-value**  
Report the mean difference, 95% CI, and Cohen's d. A significant p with d=0.08 describes a detectable but negligible effect.

**3. Using a two-sample test on paired data**  
Paired data has within-pair correlation. Ignoring it with an independent test discards statistical power. Use `ttest_rel()`.

**4. Running Levene's test first to choose the t-test variant**  
This two-stage procedure inflates Type I error. Always use Welch unconditionally.

**5. Treating non-significant results as evidence of no difference**  
Report the CI for the difference; wide CI means underpowered, not equivalent groups.
---
*python_methods_library - Samantha McGarrigle*